In [ ]:
import os
import re
import glob
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime
from multiprocessing import Pool
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
MD_FILES_PATH = os.path.expanduser("~/Desktop/SEC_AI_AGENT/data/md_files")
DB_CONFIG = {
    "host": "localhost",
    "database": "sec_filings",
    "user": "ashish",
    "password": "ashish"
}
MAX_TOKENS    = 1024   # words per chunk
OVERLAP_TOKENS = 200   # word overlap between consecutive chunks
BATCH_SIZE     = 500   # rows per PostgreSQL insert batch
EMBED_BATCH_SIZE = 256 # chunks per GPU encode call
CPU_WORKERS    = 24    # parallel processes for chunking

In [ ]:
# ─── Parse metadata from filename ─────────────────────────────────────────────
def parse_filename(filepath):
    """Extract ticker, filing_type, date from filename like AAPL_10-K_2024-01-15.md"""
    filename = os.path.basename(filepath).replace(".md", "")
    parts = filename.split("_")

    if len(parts) >= 3:
        ticker = parts[0]
        filing_type = parts[1]
        date_str = parts[2]
        try:
            filing_date = datetime.strptime(date_str, "%Y-%m-%d").date()
        except ValueError:
            filing_date = None
    elif len(parts) == 2:
        ticker = parts[0]
        filing_type = parts[1]
        filing_date = None
    else:
        ticker = filename
        filing_type = "unknown"
        filing_date = None

    return ticker, filing_type, filing_date

In [ ]:
# ─── Split text by headers ────────────────────────────────────────────────────
def split_by_sections(text):
    """Split text by markdown headers (# ## ###). Returns list of (section_name, text)."""
    pattern = r'(^#{1,3}\s+.+$)'
    parts = re.split(pattern, text, flags=re.MULTILINE)

    sections = []
    current_section = "Introduction"
    current_text = ""

    for part in parts:
        part = part.strip()
        if not part:
            continue
        if re.match(r'^#{1,3}\s+', part):
            if current_text.strip():
                sections.append((current_section, current_text.strip()))
            current_section = re.sub(r'^#{1,3}\s+', '', part).strip()
            current_text = ""
        else:
            current_text += part + "\n"

    if current_text.strip():
        sections.append((current_section, current_text.strip()))

    if not sections:
        sections = [("Full Document", text.strip())]

    return sections

In [ ]:
# ─── Chunk text into overlapping windows ──────────────────────────────────────
def chunk_text(text):
    """
    Split text into overlapping word-based chunks.
    Window size = MAX_TOKENS words, stride = MAX_TOKENS - OVERLAP_TOKENS words.
    """
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + MAX_TOKENS, len(words))
        chunks.append(' '.join(words[start:end]))
        if end >= len(words):
            break
        start += MAX_TOKENS - OVERLAP_TOKENS

    return chunks

In [ ]:
# ─── Process a single file (runs in parallel) ─────────────────────────────────
def process_file(filepath):
    """Read file, split into sections, chunk large sections. Returns list of chunk dicts."""
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()

        if not text.strip():
            return []

        ticker, filing_type, filing_date = parse_filename(filepath)
        sections = split_by_sections(text)

        chunks = []
        for section_name, section_text in sections:
            for idx, chunk in enumerate(chunk_text(section_text)):
                if len(chunk.strip()) < 50:
                    continue
                chunks.append({
                    "ticker":      ticker,
                    "filing_type": filing_type,
                    "filing_date": filing_date,
                    "section":     section_name[:500],
                    "chunk_index": idx,
                    "chunk_text":  chunk,
                    "source_file": os.path.basename(filepath)
                })
        return chunks

    except Exception as e:
        print(f"ERROR processing {filepath}: {e}")
        return []

In [ ]:
# ─── 1. Scan markdown files ───────────────────────────────────────────────────
print("Scanning for markdown files...")
all_files = glob.glob(os.path.join(MD_FILES_PATH, "**", "*.md"), recursive=True)
print(f"Found {len(all_files)} files\n")

# ─── 2. Chunk all files in parallel ──────────────────────────────────────────
print(f"Chunking with {CPU_WORKERS} workers...")
all_chunks = []
with Pool(processes=CPU_WORKERS) as pool:
    for file_chunks in tqdm(
        pool.imap_unordered(process_file, all_files),
        total=len(all_files), desc="Chunking"
    ):
        all_chunks.extend(file_chunks)

print(f"\nTotal chunks: {len(all_chunks):,}\n")

# ─── 3. Generate embeddings on GPU ───────────────────────────────────────────
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

print(f"Generating embeddings (batch size {EMBED_BATCH_SIZE})...")
chunk_texts = [c["chunk_text"] for c in all_chunks]
all_embeddings = []
for i in tqdm(range(0, len(chunk_texts), EMBED_BATCH_SIZE), desc="Embedding"):
    batch = chunk_texts[i:i + EMBED_BATCH_SIZE]
    embeddings = model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
    all_embeddings.extend(embeddings)

print(f"Generated {len(all_embeddings):,} embeddings\n")

# ─── 4. Insert into PostgreSQL ────────────────────────────────────────────────
print("Inserting into PostgreSQL...")
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# Clear existing data for a clean run
cur.execute("TRUNCATE TABLE filing_chunks RESTART IDENTITY;")
conn.commit()

insert_sql = """
    INSERT INTO filing_chunks
    (ticker, filing_type, filing_date, section, chunk_index, chunk_text, embedding, source_file)
    VALUES %s
"""

rows = [
    (
        c["ticker"], c["filing_type"], c["filing_date"],
        c["section"], c["chunk_index"], c["chunk_text"],
        emb.tolist(), c["source_file"]
    )
    for c, emb in zip(all_chunks, all_embeddings)
]

for i in tqdm(range(0, len(rows), BATCH_SIZE), desc="Inserting"):
    execute_values(cur, insert_sql, rows[i:i + BATCH_SIZE])
    conn.commit()

# ─── 5. Create vector index AFTER bulk insert ─────────────────────────────────
print("\nCreating vector search index (may take a few minutes)...")
cur.execute("""
    DROP INDEX IF EXISTS idx_embedding;
    CREATE INDEX idx_embedding ON filing_chunks
    USING ivfflat (embedding vector_cosine_ops)
    WITH (lists = 100);
""")
conn.commit()
cur.close()
conn.close()

print(f"\nDone! {len(rows):,} chunks indexed in PostgreSQL.")
print("Query example:")
print("  SELECT * FROM filing_chunks ORDER BY embedding <=> '[your_vector]' LIMIT 10;")

In [ ]:
# ─── Verify ───────────────────────────────────────────────────────────────────
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

cur.execute("SELECT COUNT(*) FROM filing_chunks;")
total = cur.fetchone()[0]
print(f"Total rows in DB: {total:,}")

cur.execute("""
    SELECT ticker, filing_type, COUNT(*) AS chunks
    FROM filing_chunks
    GROUP BY ticker, filing_type
    ORDER BY chunks DESC
    LIMIT 10;
""")
print("\nTop 10 tickers by chunk count:")
for row in cur.fetchall():
    print(f"  {row[0]:6s}  {row[1]:6s}  {row[2]:,} chunks")

cur.close()
conn.close()